In [ ]:
# Show the Python version and check whether a GPU is attached
import sys
print("Python version:", sys.version.split()[0])

# nvidia-smi lists GPUs. On a CPU runtime it fails, so we print a friendly message instead
!nvidia-smi -L || echo "No GPU - CPU runtime (this is fine for Phase 0)"

In [ ]:
# Connect your Google Drive to this Colab machine
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Use pathlib (like you learned) to point to your Drive's main folder
from pathlib import Path

MY_DRIVE = Path("/content/drive/MyDrive")
print("Drive found:", MY_DRIVE.exists())

# Show the first 10 items in your Drive, to confirm it is really your Drive
print([p.name for p in MY_DRIVE.iterdir()][:10])

In [ ]:
from pathlib import Path

# One fixed home for all projects in Drive (no spaces in the name, which is safer for shell commands)
PROJECTS_DIR = Path("/content/drive/MyDrive/projects")

# parents=True: also create any missing parent folders
# exist_ok=True: no error if the folder is already there, so it is safe to run again
PROJECTS_DIR.mkdir(parents=True, exist_ok=True)

print("Projects folder:", PROJECTS_DIR)
print("Exists:", PROJECTS_DIR.exists())
print("Contents:", [p.name for p in PROJECTS_DIR.iterdir()])

In [ ]:
# Read the token from Colab Secrets. We NEVER print the token itself.
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')

# Only print safe facts about it
print("Token loaded:", token is not None)
print("Looks like a fine-grained token:", token.startswith("github_pat_"))
print("Length:", len(token))

del token  # remove it from memory; we will load it again only when we need it

In [ ]:
from pathlib import Path

GITHUB_USER = "capzos"
REPO_NAME   = "vehicle-detection-system"

PROJECTS_DIR = Path("/content/drive/MyDrive/projects")
REPO_DIR     = PROJECTS_DIR / REPO_NAME

# A plain URL with NO token in it. This URL gets saved in the repo's settings, so it must stay clean.
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Clone only the first time. On later sessions the folder already exists in Drive.
if REPO_DIR.exists():
    print("Repo already in Drive, skipping clone:", REPO_DIR)
else:
    !git clone {REPO_URL} "{REPO_DIR}"

In [ ]:
# Make the repo folder our working folder
%cd {REPO_DIR}

# Drive does not keep Linux file permissions well. Without this line, Git may think
# every file was "modified". This setting tells Git to ignore permission changes.
!git config core.fileMode false

!git status          # should be clean
!git remote -v       # shows where we push/pull. Check that there is NO token here
!find . -path ./.git -prune -o -print | sort   # list all files and folders (skip .git)

In [ ]:
# No --global flag, so this is saved in the repo's own .git/config file.
# That file lives in Drive, so you will NOT need to set this again after a session reset.
!git config user.name  "Raj"
!git config user.email "rajkp1302@gmail.com"   # <-- CHANGE

# Show the settings saved in this repo
!git config --list --local

In [ ]:
import base64, subprocess
from google.colab import userdata

def git_push(branch="main"):
    """Push to GitHub. The token is used for this one command only and is never saved or printed."""
    token = userdata.get("GITHUB_TOKEN")
    basic = base64.b64encode(f"{GITHUB_USER}:{token}".encode()).decode()

    # "-c ..." gives Git a setting for THIS command only, so nothing is written to .git/config
    result = subprocess.run(
        ["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}",
         "push", "origin", branch],
        cwd=REPO_DIR, capture_output=True, text=True,
    )

    # Safety net: hide the token if it ever shows up in Git's messages
    output = (result.stdout + result.stderr).replace(token, "***").replace(basic, "***")
    print(output)
    print("✅ Push OK" if result.returncode == 0 else "❌ Push FAILED")

git_push()

In [ ]:
%%writefile src/colab_git.py
"""Small Git helpers for Colab. The token comes from Colab Secrets and is never saved or printed."""
import base64
import subprocess


def _run(args, repo_dir, hide=()):
    """Run a git command, print its output with any secrets hidden, and return True if it worked."""
    result = subprocess.run(args, cwd=repo_dir, capture_output=True, text=True)
    output = result.stdout + result.stderr
    for secret in hide:
        if secret:
            output = output.replace(secret, "***")
    print(output.strip())
    return result.returncode == 0


def git_pull(repo_dir):
    """Get new commits from GitHub. The repo is public, so no token is needed.
    --ff-only means: only update if it's simple. Never make a surprise merge."""
    ok = _run(["git", "pull", "--ff-only"], repo_dir)
    print("✅ Pull OK" if ok else "❌ Pull FAILED - read the message above")
    return ok


def git_push(repo_dir, github_user, branch="main"):
    """Push to GitHub using the token from Colab Secrets, for this one command only."""
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    basic = base64.b64encode(f"{github_user}:{token}".encode()).decode()
    ok = _run(
        ["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}",
         "push", "origin", branch],
        repo_dir, hide=(token, basic),
    )
    print("✅ Push OK" if ok else "❌ Push FAILED - read the message above")
    return ok

**SETUP Cell**

In [ ]:
# ================= SETUP CELL: run this FIRST in every session =================
import sys, shutil, subprocess
from pathlib import Path

GITHUB_USER = "capzos"     # <-- CHANGE
REPO_NAME   = "vehicle-detection-system"

# 1) Connect Google Drive
from google.colab import drive
drive.mount("/content/drive")

# 2) Project paths (one place to change them, used by every notebook)
PROJECT_ROOT  = Path("/content/drive/MyDrive/projects") / REPO_NAME
DATA_DIR      = PROJECT_ROOT / "data"
SAMPLES_DIR   = DATA_DIR / "samples"
MY_PHOTOS_DIR = DATA_DIR / "my_photos"
DATASETS_DIR  = DATA_DIR / "datasets"
MODELS_DIR    = PROJECT_ROOT / "models"
OUTPUTS_DIR   = PROJECT_ROOT / "outputs"

# 3) Clone the repo only if it is missing, then work inside it
if not PROJECT_ROOT.exists():
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git "{PROJECT_ROOT}"
%cd {PROJECT_ROOT}
!git config core.fileMode false

# 4) Let Python import our own code from src/
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
from colab_git import git_pull, git_push

# 5) Get the latest commits (for example, ones you pushed from your laptop)
git_pull(PROJECT_ROOT)

# 6) Which machine did we get?
if shutil.which("nvidia-smi"):
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
    print("GPU:", gpu)
else:
    print("GPU: none (CPU runtime)")
print("Ready. PROJECT_ROOT =", PROJECT_ROOT)